In [ ]:
import pandas as pd
from pathlib import Path

# Load Training Data

In [ ]:
# load training data
DATA_DIR = "split_data"           # same folder frozen_train.ipynb reads from
SPLIT_DIR = "frozen-split_data"   # where frozen_train.ipynb wrote the split map
OUTPUT_DIR = Path("task3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

contracts = pd.read_parquet(f"{DATA_DIR}/contracts.parquet")
documents = pd.read_parquet(f"{DATA_DIR}/documents.parquet")
categories = pd.read_parquet(f"{DATA_DIR}/categories.parquet")
annotation_sets = pd.read_parquet(f"{DATA_DIR}/annotation_sets.parquet")
spans = pd.read_parquet(f"{DATA_DIR}/spans.parquet")
split_map = pd.read_parquet(f"{SPLIT_DIR}/frozen-split.parquet")

documents = documents.merge(split_map, on="context_group_id", how="left")

# each document should have a split label
unmatched = documents["split"].isna().sum()
print(f"documents with NO split label (should be 0): {unmatched}")

train_documents = documents[documents["split"] == "train"].copy()
train_contract_ids = set(train_documents["contract_id"])

train_contracts = contracts[contracts["contract_id"].isin(train_contract_ids)].copy()
train_annotation_sets = annotation_sets[annotation_sets["contract_id"].isin(train_contract_ids)].copy()
train_spans = spans[spans["annotation_set_id"].isin(train_annotation_sets["annotation_set_id"])].copy()

print(f"train contracts:{len(train_contracts)}")
print(f"train documents:{len(train_documents)}")
print(f"train annotation sets:{len(train_annotation_sets)}")
print(f"train spans:{len(train_spans)}")

# rename references
contracts = train_contracts
documents = train_documents
categories = categories
annotation_sets = train_annotation_sets
spans = train_spans
# TODO: save all this in a parquet

# Finding Positive Examples
Using training data only, report the number of positive Contracts, distinct context groups, and annotated spans for each category.

In [ ]:
# category names from task 5... TODO: verify these categories
final_cat_names = [
    "Governing Law",
    "Renewal Term",
    "Revenue/Profit Sharing",
    "Cap On Liability",
    "Uncapped Liability",
    "Termination For Convenience",
    "Anti-Assignment",
    "Audit Rights",
    "License Grant",
    "Exclusivity"
]
final_ids = categories.loc[categories.category_name.isin(final_cat_names), 'category_id']

# find the annotated sets that are related to each of the final categories
condition = annotation_sets.category_id.isin(final_ids) & ~annotation_sets.is_impossible
positive_sets = annotation_sets[condition]
positive_sets

In [ ]:
# contracts that contain the positive sets
positive_contracts = positive_sets.merge(documents[['document_id','contract_id', 'context_group_id']], on="contract_id")

summary = positive_contracts.groupby('category_id').agg(
    num_positive_contracts=('contract_id', 'nunique'),
    num_positive_context_groups=('context_group_id', 'nunique'),
)

# num annotated spans for each category
positive_spans = spans.merge(positive_sets[["annotation_set_id", "category_id"]], on="annotation_set_id")
span_count = positive_spans.groupby('category_id').size().reset_index(name="num_positive_span")

summary = summary.merge(span_count, on="category_id")

**Final Summary**
The number of positive contracts, distinct context groups, and annotated spans for each category

In [ ]:
summary

 Confirm that each category has at least 20 positive Contracts and 20 positive context groups. Document and justify any exception.

Each category has sufficient examples of positive contracts and context groups.

In [ ]:
(summary["num_positive_contracts"] < 20).any() | (summary["num_positive_context_groups"] < 20).any()

# Category Co-Occurrence
Show how often related categories appear together.

In [ ]:
presence = positive_contracts[["contract_id", "category_id"]].drop_duplicates()
# contract x category 0/1 matrix, keeping all 41 category columns even if some have zero train support
presence_matrix = (
    presence.assign(present=1)
    .pivot_table(index="contract_id", columns="category_id", values="present", fill_value=0)
    .reindex(columns=final_ids, fill_value=0)
)

# cooccurrence[i, j] = number of contracts where category i AND category j both appear
cooccurrence = presence_matrix.T.dot(presence_matrix)
cooccurrence

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(cooccurrence.values, cmap="viridis")
ax.set_xticks(range(len(cooccurrence.columns)))
ax.set_xticklabels(cooccurrence.columns, rotation=90, fontsize=10)
ax.set_yticks(range(len(cooccurrence.index)))
ax.set_yticklabels(cooccurrence.index, fontsize=10)
ax.set_title("Category co-occurrence (train contracts)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# Hard Examples
 Provide five representative positive source-text excerpts and five realistic hard-negative source-text excerpts from the training Contracts for each category. For every excerpt, record its source Contract and text location. For each positive excerpt, also identify the annotated Span.

## Positive Examples

In [ ]:
pos_span_w_contract = positive_spans.merge(
    positive_sets[['annotation_set_id', 'contract_id']],
    on='annotation_set_id'
)

# 5 random examples per group, seeded so "randomness" persists
positive_examples = pos_span_w_contract[["contract_id", "annotation_set_id", "category_id", "answer_text", "answer_start", "answer_end"]].groupby("category_id").sample(n=5, random_state=42)

positive_examples

In [ ]:
display_samples = positive_examples.sample(n=5, random_state=42)
for _, sample in display_samples.iterrows():
    print(f"CATEGORY ID: {sample['category_id']}")
    print(f"CONTRACT ID: {sample['contract_id']}")
    print(f"EXAMPLE: {sample['answer_text']}\n")

## Negative Examples

In [ ]:
# find the annotated sets that are related to each of the final categories
condition = annotation_sets.category_id.isin(final_ids) & annotation_sets.is_impossible
negative_sets = annotation_sets[condition]
negative_sets

In [ ]:
all_spans_w_category = spans.merge(annotation_sets[["annotation_set_id", "category_id", "contract_id"]], on="annotation_set_id")
negative_spans = all_spans_w_category[~all_spans_w_category['span_id'].isin(positive_spans['span_id'])]

Confirm that there are no duplicates (both positive and negative examples)

In [ ]:
negative_spans.shape[0] + positive_spans.shape[0] == spans.shape[0]

In [ ]:
# 5 random examples per group, seeded so "randomness" persists
negative_examples = negative_spans[["contract_id", "annotation_set_id", "category_id", "answer_text", "answer_start", "answer_end"]].groupby("category_id").sample(n=5, random_state=42)
negative_examples

## Analysis

In [ ]:
try:
    task6_bundles = pd.read_csv(Path("task6_outputs") / "proposed_bundles.csv")
    bundle_partner = dict(zip(task6_bundles["category_a"], task6_bundles["category_b"]))
    bundle_partner.update(dict(zip(task6_bundles["category_b"], task6_bundles["category_a"])))
    print("loaded bundle pairing from task6_outputs/proposed_bundles.csv")
except FileNotFoundError:
    bundle_partner = {
        "Cap On Liability": "Uncapped Liability", "Uncapped Liability": "Cap On Liability",
        "License Grant": "Exclusivity", "Exclusivity": "License Grant",
        "Renewal Term": "Termination For Convenience", "Termination For Convenience": "Renewal Term",
        "Revenue/Profit Sharing": "Audit Rights", "Audit Rights": "Revenue/Profit Sharing",
        "Governing Law": "Anti-Assignment", "Anti-Assignment": "Governing Law",
    }
    print("task6_outputs/proposed_bundles.csv not found -- using Task #6's draft pairing (update once finalized)")

In [ ]:
# id_to_name / name_to_id_map, built from the real categories table (not reconstructed by
# hand) so they match the real category_id values exactly, including "Anti-Assignment"
id_to_name = dict(zip(categories.category_id, categories.category_name))
name_to_id_map = {v: k for k, v in id_to_name.items()}

span_counts = (
    positive_spans.merge(positive_sets[["annotation_set_id", "contract_id"]], on="annotation_set_id")
    .groupby(["category_id", "contract_id"])
    .size()
    .reset_index(name="span_count")
)
span_counts.head()

In [ ]:
# real annotated spans, already restricted to the final 10 categories (positive_spans),
# with contract_id attached (pos_span_w_contract, built in the Positive Examples section above)
candidate_pool = pos_span_w_contract

def hard_negatives_for(target_name, n=5, seed=42):
    target_id = name_to_id_map[target_name]
    pool = candidate_pool[candidate_pool["category_id"] != target_id]

    picks = []
    partner_name = bundle_partner.get(target_name)
    partner_id = name_to_id_map.get(partner_name) if partner_name else None
    if partner_id is not None:
        partner_pool = pool[pool["category_id"] == partner_id]
        n_partner = min(3, len(partner_pool))
        if n_partner:
            picks.append(partner_pool.sample(n=n_partner, random_state=seed))

    chosen_so_far = pd.concat(picks) if picks else pool.iloc[0:0]
    remaining = n - len(chosen_so_far)
    if remaining > 0:
        rest_pool = pool[~pool.index.isin(chosen_so_far.index)]
        picks.append(rest_pool.sample(n=min(remaining, len(rest_pool)), random_state=seed))

    result = pd.concat(picks).reset_index(drop=True)
    result["target_category"] = target_name
    result["negative_source_category"] = result["category_id"].map(id_to_name)
    return result

hard_negative_examples = pd.concat(
    [hard_negatives_for(name) for name in final_cat_names],
    ignore_index=True,
)
hard_negative_examples

In [ ]:
counts = hard_negative_examples.groupby("target_category").size()
checks = []
if not (counts == 5).all():
    checks.append(f"expected 5 hard negatives per category, got: {dict(counts[counts != 5])}")
if (hard_negative_examples["negative_source_category"] == hard_negative_examples["target_category"]).any():
    checks.append("a hard negative was drawn from its own target category")
if set(hard_negative_examples["target_category"]) != set(final_cat_names):
    checks.append("not every final-10 category got hard negatives")

if checks:
    print("FAILED:")
    for c in checks:
        print(f"  - {c}")
else:
    print("PASSED -- 5 hard negatives per category, all drawn from a different final-10 category")

In [ ]:
hard_negative_examples.to_csv(OUTPUT_DIR / "hard_negative_examples.csv", index=False)
hard_negative_examples[["target_category", "negative_source_category", "contract_id", "answer_text"]].groupby("target_category").head(2)

In [ ]:
# category_descriptions.csv has the official CUAD text (name/description/answer format/group);
# note it's case-inconsistent with our final_cat_names ("Cap on Liability" vs "Cap On Liability"),
# so match case-insensitively rather than on exact string equality
desc_by_name = {}
try:
    cat_desc = pd.read_csv("../data/cuad/category_descriptions.csv", encoding="utf-8-sig")
    cat_desc["name_clean"] = cat_desc["Category (incl. context and answer)"].str.replace("Category: ", "", regex=False)
    cat_desc["description_clean"] = cat_desc["Description"].str.replace("Description: ", "", regex=False)
    cat_desc["group_clean"] = cat_desc["Group"].str.replace("Group: ", "", regex=False)
    desc_by_name = {
        row["name_clean"].strip().lower(): {"description": row["description_clean"], "cuad_group": row["group_clean"]}
        for _, row in cat_desc.iterrows()
    }
    print(f"loaded {len(desc_by_name)} category descriptions from category_descriptions.csv")
except FileNotFoundError:
    print("category_descriptions.csv not found in the working directory -- evidence pack will skip descriptions")

# Category Concentration

For each of the final 10 categories: how many *different* contracts is the positive signal
actually coming from? A category with hundreds of spans that are mostly from a handful of
contracts is a lot thinner than the raw span count suggests.

In [ ]:
def concentration_stats(group):
    total = group["span_count"].sum()
    sorted_counts = group["span_count"].sort_values(ascending=False).to_numpy()
    shares = sorted_counts / total
    return pd.Series({
        "num_contracts": len(group),
        "total_spans": int(total),
        "spans_per_contract": total / len(group),
        "top1_contract_share": shares[0],
        "top3_contract_share": shares[:3].sum(),
        "hhi": (shares ** 2).sum(),
    })

CONCENTRATION_FLAG_THRESHOLD = 0.25

concentration = span_counts.groupby("category_id").apply(concentration_stats).reset_index()
concentration["category_name"] = concentration["category_id"].map(id_to_name)
concentration["flag_single_contract_heavy"] = concentration["top1_contract_share"] > CONCENTRATION_FLAG_THRESHOLD
concentration = concentration.sort_values("top1_contract_share", ascending=False).reset_index(drop=True)

concentration.to_csv(OUTPUT_DIR / "category_concentration.csv", index=False)
concentration

In [ ]:
import json

category_evidence = {}
summary_rows = []

for name in final_cat_names:
    cid = name_to_id_map[name]
    stats_row = summary[summary["category_id"] == cid].iloc[0]
    conc_row = concentration[concentration["category_id"] == cid].iloc[0]
    desc = desc_by_name.get(name.strip().lower(), {})

    # top 3 categories this one co-occurs with most, from the real cooccurrence matrix above
    co_series = cooccurrence[cid].drop(index=cid).sort_values(ascending=False)
    top_related = [
        {"category": id_to_name[other_id], "cooccurrence_count": int(count)}
        for other_id, count in co_series.head(3).items()
    ]

    pos_examples = positive_examples.loc[
        positive_examples["category_id"] == cid,
        ["contract_id", "answer_text", "answer_start", "answer_end"],
    ].to_dict(orient="records")

    neg_examples = hard_negative_examples.loc[
        hard_negative_examples["target_category"] == name,
        ["contract_id", "negative_source_category", "answer_text", "answer_start", "answer_end"],
    ].to_dict(orient="records")

    category_evidence[name] = {
        "category_id": cid,
        "cuad_group": desc.get("cuad_group"),
        "description": desc.get("description"),
        "statistics": {
            "num_positive_contracts": int(stats_row["num_positive_contracts"]),
            "num_positive_context_groups": int(stats_row["num_positive_context_groups"]),
            "num_positive_spans": int(stats_row["num_positive_span"]),
            "spans_per_contract": round(float(conc_row["spans_per_contract"]), 2),
            "top1_contract_share": round(float(conc_row["top1_contract_share"]), 3),
            "flag_single_contract_heavy": bool(conc_row["flag_single_contract_heavy"]),
        },
        "related_categories": top_related,
        "positive_examples": pos_examples,
        "hard_negative_examples": neg_examples,
    }

    summary_rows.append({
        "category_name": name,
        "cuad_group": desc.get("cuad_group"),
        "num_positive_contracts": int(stats_row["num_positive_contracts"]),
        "num_positive_spans": int(stats_row["num_positive_span"]),
        "spans_per_contract": round(float(conc_row["spans_per_contract"]), 2),
        "top1_contract_share": round(float(conc_row["top1_contract_share"]), 3),
        "flag_single_contract_heavy": bool(conc_row["flag_single_contract_heavy"]),
        "top_related_category": top_related[0]["category"] if top_related else None,
        "top_related_cooccurrence": top_related[0]["cooccurrence_count"] if top_related else None,
    })

with open(OUTPUT_DIR / "category_evidence.json", "w") as f:
    json.dump(category_evidence, f, indent=2)

evidence_summary = pd.DataFrame(summary_rows)
evidence_summary.to_csv(OUTPUT_DIR / "category_evidence_summary.csv", index=False)

evidence_summary

In [ ]:
checks = []
if len(category_evidence) != 10:
    checks.append(f"expected 10 categories in the evidence pack, got {len(category_evidence)}")
for name, entry in category_evidence.items():
    if len(entry["positive_examples"]) != 5:
        checks.append(f"{name}: expected 5 positive examples, got {len(entry['positive_examples'])}")
    if len(entry["hard_negative_examples"]) != 5:
        checks.append(f"{name}: expected 5 hard negatives, got {len(entry['hard_negative_examples'])}")
    if entry["description"] is None:
        checks.append(f"{name}: no CUAD description matched -- check category_descriptions.csv")

if checks:
    print("FAILED:")
    for c in checks:
        print(f"  - {c}")
else:
    print("PASSED -- all 10 categories have a description, statistics, related categories, 5 positive examples, and 5 hard negatives")